In [1]:
"""
train_eval_v11_DualDDT.py  —  AE puro + DDT Independiente Doble
===============================================================
Metodología de "Caja Gris" (Grey-Box Tagger):
  1. Espacio de Fase Limpio: pT > 250 GeV y SDMass en [40, 200] GeV.
  2. AE entrenado SOLO con Chamfer para aprender topología sin sesgos.
  3. Purificación Independiente (Dual DDT):
     - C_DDT = Chamfer - p1(ρ)
     - τ_DDT = τ₂₁ - p2(ρ)
  4. Optimización (Grid Search):
     - Se busca maximizar el AUC de la combinación final: 
       s = α·C_DDT - γ·τ_DDT
  5. Resultado: Tagger híbrido veloz, robusto y 100% decorrelado.
"""

import os, json, glob, torch, resource, itertools
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import multiprocessing
from sklearn.metrics import roc_auc_score, roc_curve
from scipy.stats import pearsonr
from numpy.polynomial.chebyshev import Chebyshev

import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning, message="divide by zero encountered in divide")

# ==============================================================================
# 0. CONFIGURACIÓN
# ==============================================================================
def configure_system_resources(max_ram_gb=100, target_cores=14):
    try:
        limit = int(max_ram_gb * 1024**3)
        soft, hard = resource.getrlimit(resource.RLIMIT_AS)
        resource.setrlimit(resource.RLIMIT_AS, (limit, hard))
    except Exception as e:
        print(f"[!] RAM limit: {e}")
    use_cores = min(target_cores, multiprocessing.cpu_count())
    torch.set_num_threads(use_cores)
    os.environ["OMP_NUM_THREADS"] = str(use_cores)

def fast_parse(df_col, max_p):
    df_col   = df_col.str.replace('"', '', regex=False)
    expanded = (df_col.str.split(';', expand=True)
                      .iloc[:, :max_p]
                      .astype(np.float32)
                      .fillna(0.0))
    if expanded.shape[1] < max_p:
        pad = np.zeros((len(expanded), max_p - expanded.shape[1]), dtype=np.float32)
        return np.hstack([expanded.values, pad])
    return expanded.values

def jet_axis(pt, eta, phi):
    s  = np.sum(pt, axis=1, keepdims=True) + 1e-8
    we = np.sum(pt * eta,         axis=1, keepdims=True) / s
    wx = np.sum(pt * np.cos(phi), axis=1, keepdims=True)
    wy = np.sum(pt * np.sin(phi), axis=1, keepdims=True)
    return we, np.arctan2(wy, wx)

# ==============================================================================
# 1. PREPROCESADOR  (Aplicando Cortes Cinemáticos)
# ==============================================================================
class PrepV10:
    def __init__(self, max_p=30):
        self.max_p     = max_p
        self.feat_mean = None
        self.feat_std  = None

    def fit(self, files):
        print(f"[*] Fit preprocesador sobre {len(files)} archivo(s)...")
        all_deta, all_dphi, all_logpt = [], [], []
        
        for f in files[:2]:
            df  = pd.read_csv(f, usecols=['PF_Pt','PF_Eta','PF_Phi', 'SDMass'], nrows=60000)
            
            pt  = fast_parse(df['PF_Pt'],  self.max_p)
            eta = fast_parse(df['PF_Eta'], self.max_p)
            phi = fast_parse(df['PF_Phi'], self.max_p)
            
            px, py = pt*np.cos(phi), pt*np.sin(phi)
            jpt = np.sqrt(px.sum(1)**2 + py.sum(1)**2)
            sdm = df['SDMass'].values.astype(np.float32)
            
            # --- CORTES CINEMÁTICOS ---
            valid_mask = (jpt > 250.0) & (sdm >= 40.0) & (sdm <= 200.0)
            
            pt  = pt[valid_mask]
            eta = eta[valid_mask]
            phi = phi[valid_mask]
            
            je, jp = jet_axis(pt, eta, phi)
            mask = pt > 0
            
            all_deta.append((eta - je)[mask])
            all_dphi.append(((phi - jp + np.pi) % (2*np.pi) - np.pi)[mask])
            all_logpt.append(np.log1p(pt)[mask])
            
        deta  = np.concatenate(all_deta)
        dphi  = np.concatenate(all_dphi)
        logpt = np.concatenate(all_logpt)
        
        self.feat_mean = np.array([deta.mean(), dphi.mean(), logpt.mean()], dtype=np.float32)
        self.feat_std  = np.array([deta.std(),  dphi.std(),  logpt.std()],  dtype=np.float32)
        print(f"    feat_mean={self.feat_mean}")

    def save(self, path="scaler_v11.json"):
        with open(path, "w") as f:
            json.dump({"feat_mean": self.feat_mean.tolist(),
                       "feat_std":  self.feat_std.tolist()}, f)

    def load(self, path="scaler_v11.json"):
        with open(path) as f:
            d = json.load(f)
        self.feat_mean = np.array(d["feat_mean"], dtype=np.float32)
        self.feat_std  = np.array(d["feat_std"],  dtype=np.float32)

# ==============================================================================
# 2. DATASET (Aplicando Cortes Cinemáticos)
# ==============================================================================
class JetDatasetV10(Dataset):
    def __init__(self, file_path, prep, max_p=30, force=False, max_rows=None):
        self.cache = file_path.replace(".csv", "_cache_v11.pt")
        if force and os.path.exists(self.cache):
            os.remove(self.cache)

        if os.path.exists(self.cache) and not force:
            self.feat, self.mask, self.phys = torch.load(self.cache, weights_only=False)
            if max_rows and len(self.feat) > max_rows:
                self.feat = self.feat[:max_rows]
                self.mask = self.mask[:max_rows]
                self.phys = self.phys[:max_rows]
            return

        df  = pd.read_csv(file_path, nrows=max_rows)
        pt  = fast_parse(df['PF_Pt'],  max_p)
        eta = fast_parse(df['PF_Eta'], max_p)
        phi = fast_parse(df['PF_Phi'], max_p)

        px, py = pt*np.cos(phi), pt*np.sin(phi)
        pz, e  = pt*np.sinh(eta), pt*np.cosh(eta)
        jm2    = e.sum(1)**2 - px.sum(1)**2 - py.sum(1)**2 - pz.sum(1)**2
        jmass  = np.sqrt(np.maximum(0, jm2))
        jpt    = np.sqrt(px.sum(1)**2 + py.sum(1)**2)
        sdm    = df['SDMass'].values.astype(np.float32)

        # --- APLICACIÓN DE CORTES CINEMÁTICOS ---
        valid_mask = (jpt > 250.0) & (sdm >= 40.0) & (sdm <= 200.0)
        
        pt, eta, phi = pt[valid_mask], eta[valid_mask], phi[valid_mask]
        jmass, jpt, sdm = jmass[valid_mask], jpt[valid_mask], sdm[valid_mask]
        df = df[valid_mask]

        tau1   = df['Tau1'].values.astype(np.float32) + 1e-8
        tau2   = df['Tau2'].values.astype(np.float32)
        tau21  = tau2 / tau1
        sig    = (df['is_signal'].values.astype(np.float32)
                  if 'is_signal' in df.columns else np.zeros(len(df), np.float32))
        nc     = np.sum(pt > 0, 1).astype(np.float32)

        rho = np.log(np.clip(sdm**2 / (jpt**2 + 1e-6), 1e-6, None)).astype(np.float32)
        phys_arr = np.stack([jmass, jpt, nc, tau21, sdm, sig, rho], axis=1).astype(np.float32)

        je, jp = jet_axis(pt, eta, phi)
        deta   = eta - je
        dphi   = (phi - jp + np.pi) % (2*np.pi) - np.pi
        logpt  = np.log1p(pt)
        masks  = (pt > 0).astype(np.float32)

        dn = (deta  - prep.feat_mean[0]) / (prep.feat_std[0]+1e-8) * masks
        dp = (dphi  - prep.feat_mean[1]) / (prep.feat_std[1]+1e-8) * masks
        dl = (logpt - prep.feat_mean[2]) / (prep.feat_std[2]+1e-8) * masks

        feat_arr = np.stack([dn, dp, dl], axis=2).astype(np.float32)

        self.feat = torch.from_numpy(feat_arr)
        self.mask = torch.from_numpy(masks)
        self.phys = torch.from_numpy(phys_arr)
        torch.save((self.feat, self.mask, self.phys), self.cache)

    def __len__(self):            return len(self.feat)
    def __getitem__(self, idx):  return self.feat[idx], self.mask[idx], self.phys[idx]

# ==============================================================================
# 3 & 4. CHAMFER DISTANCE Y MODELO
# ==============================================================================
def chamfer(p1, p2, m1, m2, reduction='mean'):
    d  = torch.cdist(p1, p2, p=2)**2
    d  = d + (1-m1.unsqueeze(2))*1e8 + (1-m2.unsqueeze(1))*1e8
    pj = (d.min(2)[0]*m1).sum(1) + (d.min(1)[0]*m2).sum(1)
    pj = pj / (m1.sum(1) + 1e-8)
    return pj if reduction == 'none' else pj.mean()

class EdgeConv(nn.Module):
    def __init__(self, in_ch, out_ch, k=16):
        super().__init__()
        self.k    = k
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch*2, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.LeakyReLU(0.2),
        )
        self.skip = (nn.Conv1d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity())

    def forward(self, x):
        b, c, n = x.shape
        k       = min(self.k, n)
        inner   = -2*torch.bmm(x.transpose(1,2), x)
        xx      = x.pow(2).sum(1, keepdim=True)
        dist    = -(xx + inner + xx.transpose(1,2))
        idx     = dist.topk(k, -1)[1]
        base    = torch.arange(b, device=x.device).view(-1,1,1)*n
        flat    = x.transpose(1,2).reshape(b*n, c)
        nbrs    = flat[(idx+base).view(-1)].view(b,n,k,c).permute(0,3,1,2)
        xc      = x.unsqueeze(3).expand_as(nbrs)
        return self.conv(torch.cat([xc, nbrs-xc], 1)).mean(3) + self.skip(x)

class JetAE_V10(nn.Module):
    def __init__(self, n_part=30, latent=64, ch=128):
        super().__init__()
        self.n_part = n_part
        self.e1 = EdgeConv(3,     ch//2, k=16)
        self.e2 = EdgeConv(ch//2, ch,    k=16)
        self.e3 = EdgeConv(ch,    ch,    k=16)
        self.fc_z = nn.Sequential(
            nn.Linear(ch*2, latent*2), nn.BatchNorm1d(latent*2), nn.LeakyReLU(0.2),
            nn.Linear(latent*2, latent),
        )
        self.dec = nn.Sequential(
            nn.Linear(latent, 256), nn.BatchNorm1d(256), nn.GELU(),
            nn.Linear(256,    512), nn.BatchNorm1d(512), nn.GELU(),
            nn.Linear(512, n_part*3),
        )

    def encode(self, x):
        h = self.e1(x.transpose(1,2))
        h = self.e2(h)
        h = self.e3(h)
        return self.fc_z(torch.cat([h.max(2)[0], h.mean(2)], 1))

    def forward(self, x):
        z     = self.encode(x)
        recon = self.dec(z).view(-1, self.n_part, 3)
        return recon, z

# ==============================================================================
# 5 & 6. ENTRENAMIENTO Y RECOLECCIÓN
# ==============================================================================
def train_ae(model, tr_loader, va_loader, device, num_epochs=60, lr=1e-3, save_path="best_ae_v11.pt"):
    opt  = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=num_epochs, eta_min=1e-5)
    best_val = float('inf')
    history  = {'train': [], 'val': []}

    for epoch in range(num_epochs):
        model.train()
        run = 0.0
        pbar = tqdm(tr_loader, desc=f"Epoch {epoch+1:>3}/{num_epochs}", ncols=100)
        for feat, mask, phys in pbar:
            feat = feat.to(device); mask = mask.to(device)
            opt.zero_grad()
            recon, _ = model(feat)
            loss = chamfer(feat, recon, mask, mask)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            run += loss.item()
            pbar.set_postfix(Chamfer=f"{loss.item():.5f}")

        avg_tr = run / len(tr_loader)
        history['train'].append(avg_tr)

        model.eval()
        val_l = 0.0
        with torch.no_grad():
            for feat, mask, phys in va_loader:
                feat = feat.to(device); mask = mask.to(device)
                recon, _ = model(feat)
                val_l += chamfer(feat, recon, mask, mask).item()
        avg_val = val_l / len(va_loader)
        history['val'].append(avg_val)
        sched.step()

        print(f"  → Train={avg_tr:.5f}  Val={avg_val:.5f}")
        if avg_val < best_val:
            best_val = avg_val
            torch.save(model.state_dict(), save_path)
            print(f"  ✓ Checkpoint (val={best_val:.5f})")
    return history

@torch.no_grad()
def collect_raw(model, loader, device):
    model.eval()
    C_l, T_l, R_l, L_l, M_l, PT_l = [], [], [], [], [], []
    for feat, mask, phys in loader:
        feat = feat.to(device); mask = mask.to(device)
        recon, _ = model(feat)
        c = chamfer(feat, recon, mask, mask, 'none')

        C_l.extend(c.cpu().numpy())
        T_l.extend(phys[:,3].numpy())   # tau21
        R_l.extend(phys[:,6].numpy())   # rho
        L_l.extend(phys[:,5].numpy())   # is_signal
        M_l.extend(phys[:,4].numpy())   # SDMass
        PT_l.extend(phys[:,1].numpy())  # jet_pt

    return (np.array(C_l), np.array(T_l), np.array(R_l),
            np.array(L_l), np.array(M_l), np.array(PT_l))

# ==============================================================================
# 7. DDT  —  Decorrelated Decorrelation Transform (Con Chebyshev)
# ==============================================================================
class DDTransform:
    def __init__(self, name="Var", poly_deg=3, quantile=50):
        self.name       = name
        self.poly_deg   = poly_deg
        self.quantile   = quantile
        self.poly_model = None
        self.rho_mean   = 0.0
        self.rho_std    = 1.0

    def fit(self, scores_qcd, rho_qcd, n_bins=20):
        print(f"[*] Ajustando DDT para '{self.name}' (deg={self.poly_deg}, q={self.quantile})...")

        self.rho_mean = float(rho_qcd.mean())
        self.rho_std  = float(rho_qcd.std()) + 1e-8
        rho_n = (rho_qcd - self.rho_mean) / self.rho_std

        rho_edges = np.percentile(rho_n, np.linspace(0, 100, n_bins+1))
        rho_centers, quantiles = [], []

        for i in range(n_bins):
            mask = (rho_n >= rho_edges[i]) & (rho_n < rho_edges[i+1])
            if mask.sum() < 30:
                continue
            q_val = np.percentile(scores_qcd[mask], self.quantile)
            rho_centers.append(0.5*(rho_edges[i] + rho_edges[i+1]))
            quantiles.append(q_val)

        rho_centers = np.array(rho_centers)
        quantiles   = np.array(quantiles)

        self.poly_model = Chebyshev.fit(rho_centers, quantiles, self.poly_deg)
        q_pred = self.poly_model(rho_centers)
        residuals = quantiles - q_pred
        print(f"    -> Residual RMS: {residuals.std():.5f}")

        return rho_centers, quantiles, q_pred

    def transform(self, scores, rho):
        assert self.poly_model is not None, "Llama a fit() primero"
        rho_n   = (rho - self.rho_mean) / self.rho_std
        p_rho   = self.poly_model(rho_n)
        return scores - p_rho

    def save(self, path):
        with open(path,"w") as f:
            json.dump({"coeffs": self.poly_model.coef.tolist(),
                       "domain": self.poly_model.domain.tolist(),
                       "window": self.poly_model.window.tolist(),
                       "rho_mean": self.rho_mean,
                       "rho_std":  self.rho_std,
                       "poly_deg": self.poly_deg,
                       "quantile": self.quantile}, f)

    def load(self, path):
        with open(path) as f:
            d = json.load(f)
        self.poly_model = Chebyshev(d["coeffs"], domain=d["domain"], window=d["window"])
        self.rho_mean = d["rho_mean"]
        self.rho_std  = d["rho_std"]
        self.poly_deg = d["poly_deg"]
        self.quantile = d["quantile"]

# ==============================================================================
# 8. GRID SEARCH (Directo sobre Variables Purificadas)
# ==============================================================================
def fix_dir(s, labels):
    auc = roc_auc_score(labels, s)
    return (-s, 1-auc) if auc < 0.5 else (s, auc)

def grid_search_hybrid(C_ddt, T_ddt, labels):
    alphas = np.linspace(0.0, 5.0, 21)
    gammas = np.linspace(0.5, 5.0, 19) # Forzamos uso de tau21

    best_auc, best_params = 0.0, {}

    # El Grid Search ahora es instantáneo
    for alpha, gamma in itertools.product(alphas, gammas):
        # s = α * C_DDT - γ * τ_DDT
        s = alpha * C_ddt - gamma * T_ddt
        try:
            auc = roc_auc_score(labels, s)
            if auc < 0.5: auc = 1-auc
            if auc > best_auc:
                best_auc    = auc
                best_params = {'alpha': float(alpha), 'gamma': float(gamma)}
        except Exception:
            continue

    print(f"\n[+] Grid search (val) → AUC={best_auc:.4f}  "
          f"α={best_params.get('alpha', 0):.2f}  γ={best_params.get('gamma', 0):.2f}")
    return best_params, best_auc

# ==============================================================================
# 9. GRÁFICAS
# ==============================================================================
def compute_jsd(h_ref, h_cut):
    p = np.maximum(h_ref, 1e-10); p /= p.sum()
    q = np.maximum(h_cut, 1e-10); q /= q.sum()
    m = 0.5*(p+q)
    return float(0.5*np.sum(p*np.log(p/m)) + 0.5*np.sum(q*np.log(q/m)))

def plot_ddt_fit(rho_centers, quantiles, ddt_model, rho_qcd, s_raw_qcd, s_ddt_qcd, save_path):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].scatter(rho_centers, quantiles, s=40, color='steelblue', label='Cuantil por bin', zorder=5)
    rho_line = np.linspace(rho_centers.min(), rho_centers.max(), 200)
    axes[0].plot(rho_line, ddt_model.poly_model(rho_line), color='darkorange', lw=2, label='Polinomio Chebyshev')
    
    axes[0].set_xlabel('ρ normalizado')
    axes[0].set_ylabel('Cuantil del score')
    axes[0].set_title(f'Ajuste DDT para {ddt_model.name}')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].hexbin(rho_qcd, s_raw_qcd, gridsize=50, cmap='Blues', mincnt=1, bins='log')
    r, _ = pearsonr(s_raw_qcd, rho_qcd)
    axes[1].set_xlabel('ρ'); axes[1].set_ylabel(f'{ddt_model.name} Raw')
    axes[1].set_title(f'Raw vs ρ  (ρ={r:+.3f})')
    axes[1].grid(True, alpha=0.3)

    axes[2].hexbin(rho_qcd, s_ddt_qcd, gridsize=50, cmap='Greens', mincnt=1, bins='log')
    r2, _ = pearsonr(s_ddt_qcd, rho_qcd)
    axes[2].set_xlabel('ρ'); axes[2].set_ylabel(f'{ddt_model.name} DDT')
    axes[2].set_title(f'DDT vs ρ  (ρ={r2:+.3f})')
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()
    print(f"[*] DDT fit guardado en '{save_path}'")

def plot_all(C, tau21, s_raw, s_ddt, labels, masses, rho, auc, save_prefix='v11'):
    sig = labels == 1
    qcd = labels == 0

    s_dir, _      = fix_dir(s_ddt, labels)
    s_raw_dir, _  = fix_dir(s_raw, labels)

    fpr_ddt,  tpr_ddt,  thr_ddt  = roc_curve(labels, s_dir)
    fpr_raw,  tpr_raw,  _        = roc_curve(labels, s_raw_dir)
    auc_raw                       = roc_auc_score(labels, s_raw_dir)
    
    with np.errstate(divide='ignore'):
        rej_ddt = np.where(fpr_ddt > 0, 1.0/fpr_ddt, np.nan)

    idx50   = np.argmin(np.abs(tpr_ddt - 0.50))
    rej50   = rej_ddt[idx50]

    percs    = np.linspace(50, 99.5, 200)
    sig_vals, eff_s, eff_b = [], [], []
    N_s, N_b = sig.sum(), qcd.sum()
    for pct in percs:
        thr_i = np.percentile(s_dir, pct)
        ns    = np.sum(s_dir[sig] >= thr_i)
        nb    = np.sum(s_dir[qcd] >= thr_i)
        sig_vals.append(ns / np.sqrt(nb+1e-6))
        eff_s.append(ns/(N_s+1e-6)); eff_b.append(nb/(N_b+1e-6))
    best_sig_idx = np.argmax(sig_vals)

    qcd_scores = s_dir[qcd]; qcd_masses = masses[qcd]
    bins_m     = np.linspace(40, 200, 41) 
    h_ref, _   = np.histogram(qcd_masses, bins=bins_m, density=True)
    fpr_cuts   = [1.0, 0.5, 0.1, 0.05, 0.01, 0.005]
    colors_ms  = ['black','blue','orange','red','purple','magenta']
    lbls_ms    = ['100% QCD','50% QCD','10% QCD','5% QCD','1% QCD','0.5% QCD']
    jsd_vals   = {}

    fig = plt.figure(figsize=(20, 12))
    gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.38, wspace=0.32)

    ax0 = fig.add_subplot(gs[0,0])
    ax0.plot(fpr_raw, tpr_raw, color='gray',       lw=1.5, ls='--', label=f'Score raw  AUC={auc_raw:.3f}')
    ax0.plot(fpr_ddt, tpr_ddt, color='darkorange', lw=2.5, label=f'Score DDT  AUC={auc:.3f}')
    ax0.scatter([fpr_ddt[idx50]], [tpr_ddt[idx50]], s=80, color='red', zorder=5, label=f'TPR=50%  Rej={rej50:.0f}')
    ax0.plot([0,1],[0,1],'k--',lw=1,alpha=0.4)
    ax0.set_xlabel('False Positive Rate (QCD)', fontsize=11)
    ax0.set_ylabel('True Positive Rate (WW)',   fontsize=11)
    ax0.set_title('Curva ROC', fontsize=12)
    ax0.legend(fontsize=9); ax0.grid(True, alpha=0.3)

    ax1 = fig.add_subplot(gs[0,1])
    ax1.plot(tpr_ddt, rej_ddt, color='steelblue', lw=2.5)
    ax1.set_yscale('log')
    ax1.axhline(100,  color='gray', ls=':',  lw=1,   label='Rej=100')
    ax1.axhline(1000, color='gray', ls='--', lw=1,   label='Rej=1000')
    ax1.axvline(0.5,  color='red',  ls=':',  lw=1.2, label='TPR=50%')
    ax1.scatter([tpr_ddt[idx50]], [rej50], s=80, color='red', zorder=5, label=f'Rej@50%={rej50:.0f}')
    ax1.set_xlabel('True Positive Rate (WW)', fontsize=11)
    ax1.set_ylabel('Background Rejection (1/FPR)', fontsize=11)
    ax1.set_title('Background Rejection', fontsize=12)
    ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3, which='both')
    ax1.set_xlim(0,1)
    finite_rej = rej_ddt[np.isfinite(rej_ddt)]
    if len(finite_rej): ax1.set_ylim(1, finite_rej.max()*2)

    ax2 = fig.add_subplot(gs[0,2])
    bins_s = np.linspace(np.percentile(s_dir,1), np.percentile(s_dir,99), 60)
    ax2.hist(s_dir[qcd], bins=bins_s, density=True, histtype='step', lw=2, color='blue',   label='QCD (fondo)')
    ax2.hist(s_dir[sig], bins=bins_s, density=True, histtype='step', lw=2, color='red',    label='WW (señal)')
    ax2.set_xlabel('Score Combinado DDT', fontsize=11)
    ax2.set_ylabel('Densidad', fontsize=11)
    ax2.set_title('Distribución del Score DDT', fontsize=12)
    ax2.legend(fontsize=10); ax2.grid(True, alpha=0.3)

    ax3 = fig.add_subplot(gs[1,0])
    ax3b = ax3.twinx()
    ax3.plot(100-percs, sig_vals, color='darkorange', lw=2.5, label='S/√B')
    ax3b.plot(100-percs, eff_s, color='green', lw=1.5, ls='--', label='ε_señal', alpha=0.8)
    ax3b.plot(100-percs, eff_b, color='red',   lw=1.5, ls=':', label='ε_fondo', alpha=0.8)
    ax3.scatter([100-percs[best_sig_idx]], [sig_vals[best_sig_idx]], s=80, color='darkorange', zorder=5, label=f'Máx S/√B={sig_vals[best_sig_idx]:.1f}')
    ax3.set_xlabel('% QCD retenido (100% − percentil corte)', fontsize=10)
    ax3.set_ylabel('Significancia S/√B', fontsize=11)
    ax3b.set_ylabel('Eficiencia', fontsize=11)
    ax3.set_title('Significancia S/√B vs Corte', fontsize=12)
    l1,lb1 = ax3.get_legend_handles_labels()
    l2,lb2 = ax3b.get_legend_handles_labels()
    ax3.legend(l1+l2, lb1+lb2, fontsize=9, loc='upper right')
    ax3.grid(True, alpha=0.3); ax3.set_xlim(0, 50)

    ax4 = fig.add_subplot(gs[1,1])
    s_raw_q, _ = fix_dir(s_raw, labels)
    qcd_s_raw  = s_raw_q[qcd]
    for fpr_c, color, lbl in zip(fpr_cuts, colors_ms, lbls_ms):
        thr    = np.percentile(qcd_s_raw, (1.0-fpr_c)*100)
        m_pass = qcd_masses[qcd_s_raw >= thr]
        h_cut, _ = np.histogram(m_pass, bins=bins_m, density=True)
        jsd = compute_jsd(h_ref, h_cut) if fpr_c < 1.0 else None
        lbl_str = f"{lbl}  JSD={jsd:.3f}" if jsd else lbl
        ax4.hist(m_pass, bins=bins_m, density=True, histtype='step', lw=2, color=color, alpha=0.85, label=lbl_str)
    ax4.axvspan(65,95,color='gray',alpha=0.15,label='W window')
    ax4.set_xlabel('SoftDrop Mass [GeV]', fontsize=11)
    ax4.set_ylabel('Densidad Normalizada', fontsize=11)
    ax4.set_title('Mass Sculpting — Score RAW', fontsize=12)
    ax4.legend(loc='upper right', fontsize=7); ax4.grid(True, alpha=0.3)

    ax5 = fig.add_subplot(gs[1,2])
    print(f"\n  {'FPR':>6}  |  {'JSD raw':>8}  |  {'JSD DDT':>8}  |  {'N events':>8}")
    print("  " + "-"*45)
    for fpr_c, color, lbl in zip(fpr_cuts, colors_ms, lbls_ms):
        thr    = np.percentile(qcd_scores, (1.0-fpr_c)*100)
        mask_p = qcd_scores >= thr
        m_pass = qcd_masses[mask_p]
        h_cut, _ = np.histogram(m_pass, bins=bins_m, density=True)
        if fpr_c < 1.0:
            jsd = compute_jsd(h_ref, h_cut)
            jsd_vals[lbl] = jsd
            thr_r  = np.percentile(qcd_s_raw, (1.0-fpr_c)*100)
            h_r, _ = np.histogram(qcd_masses[qcd_s_raw>=thr_r], bins=bins_m, density=True)
            jsd_r  = compute_jsd(h_ref, h_r)
            print(f"  {fpr_c*100:>5.1f}%  |  {jsd_r:>8.4f}  |  {jsd:>8.4f}  |  {mask_p.sum():>8d}")
            lbl_str = f"{lbl}  JSD={jsd:.3f}"
        else:
            lbl_str = lbl
        ax5.hist(m_pass, bins=bins_m, density=True, histtype='step', lw=2, color=color, alpha=0.85, label=lbl_str)
    ax5.axvspan(65,95,color='gray',alpha=0.15,label='W window')
    ax5.set_xlabel('SoftDrop Mass [GeV]', fontsize=11)
    ax5.set_ylabel('Densidad Normalizada', fontsize=11)
    ax5.set_title('Mass Sculpting — Score DDT', fontsize=12)
    ax5.legend(loc='upper right', fontsize=7); ax5.grid(True, alpha=0.3)

    fig.suptitle(
        f'AE v11 Dual DDT  —  AUC={auc:.3f}  |  Rej@50%TPR={rej50:.0f}'
        f'  |  JSD(1%)={jsd_vals.get("1% QCD",999):.3f}',
        fontsize=14, fontweight='bold'
    )
    save_path = f'{save_prefix}_panel.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"\n[*] Panel guardado en '{save_path}'")

    print(f"\n{'='*60}")
    print(f"  AUC (DDT)               = {auc:.4f}")
    print(f"  AUC (raw, sin DDT)      = {auc_raw:.4f}")
    print(f"  Bkg Rejection @TPR=50%  = {rej50:.1f}")
    print(f"  Máx S/√B                = {sig_vals[best_sig_idx]:.2f}"
          f"  @ {100-percs[best_sig_idx]:.1f}% QCD retenido")
    print(f"  JSD por corte (DDT):")
    for lbl, jsd in jsd_vals.items():
        ok = "✓" if jsd < 0.03 else "[!]"
        print(f"    {lbl:20s}: {jsd:.4f}  {ok}")
    print(f"{'='*60}")

    return rej50, sig_vals[best_sig_idx], jsd_vals

def plot_training(history, save_path='training_history_v11.png'):
    fig, ax = plt.subplots(figsize=(8,5))
    ep = range(1, len(history['train'])+1)
    ax.plot(ep, history['train'], label='Train Chamfer', color='steelblue')
    ax.plot(ep, history['val'],   label='Val Chamfer',   color='darkorange', ls='--')
    ax.set_xlabel('Época'); ax.set_ylabel('Chamfer Loss')
    ax.set_title('Entrenamiento AE v11 (sin DisCo)')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()

# ==============================================================================
# 10. MAIN
# ==============================================================================
def main():
    configure_system_resources()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"[*] Device: {device}")

    MAX_P      = 30
    LATENT     = 64
    CHANNELS   = 128
    BATCH      = 256
    EPOCHS     = 60
    LR         = 1e-3
    MAX_ROWS   = 30000          # por archivo de entrenamiento
    TRAIN_PAT  = "file_*.csv"
    TEST_FILE  = "mixed_signal_qcd.csv"
    SCALER     = "scaler_v11.json"
    MODEL      = "best_ae_v11.pt"
    DDT_QUANT  = 50             # percentil para DDT (mediana)

    # ── Archivos de entrenamiento (Puro Fondo QCD) ────────────────────────────
    train_files = sorted(glob.glob(TRAIN_PAT))
    train_files = [f for f in train_files if 'mixed' not in f and 'signal' not in f]
    if not train_files:
        raise FileNotFoundError("No se encontraron archivos de entrenamiento.")

    # ── Preprocesador ─────────────────────────────────────────────────────────
    prep = PrepV10(MAX_P)
    if os.path.exists(SCALER):
        prep.load(SCALER); print("[*] Scaler cargado")
    else:
        prep.fit(train_files); prep.save(SCALER)

    # ── Datasets de ENTRENAMIENTO (Solo Fondo) ────────────────────────────────
    print(f"\n[*] Cargando {len(train_files)} archivo(s) para Entrenamiento AE...")
    dsets  = [JetDatasetV10(f, prep, MAX_P, force=True, max_rows=MAX_ROWS) for f in train_files]
    full   = ConcatDataset(dsets)
    
    n_tr   = int(0.9 * len(full))
    n_va   = len(full) - n_tr
    tr_ds, va_ds = torch.utils.data.random_split(full, [n_tr, n_va])
    
    tr_loader = DataLoader(tr_ds, batch_size=BATCH, shuffle=True, num_workers=4, pin_memory=True)
    va_loader = DataLoader(va_ds, batch_size=BATCH, num_workers=4, pin_memory=True)
    print(f"    AE Train={len(tr_ds)}  AE Val={len(va_ds)}")

    # ── Datasets MIXTOS (Grid Search y Test) ──────────────────────────────────
    print(f"\n[*] Cargando datos mixtos (Señal + Fondo) desde: {TEST_FILE}")
    mixed_ds = JetDatasetV10(TEST_FILE, prep, MAX_P, force=True)
    
    n_gs = int(0.5 * len(mixed_ds))
    n_te = len(mixed_ds) - n_gs
    gs_ds, te_ds = torch.utils.data.random_split(mixed_ds, [n_gs, n_te])
    
    gs_loader = DataLoader(gs_ds, batch_size=BATCH, num_workers=4, pin_memory=True)
    te_loader = DataLoader(te_ds, batch_size=BATCH, num_workers=4, pin_memory=True)
    print(f"    Grid Search={len(gs_ds)}  Test={len(te_ds)}")

    # ── Modelo ────────────────────────────────────────────────────────────────
    model = JetAE_V10(n_part=MAX_P, latent=LATENT, ch=CHANNELS).to(device)

    # ── Entrenamiento ─────────────────────────────────────────────────────────
    history = train_ae(model, tr_loader, va_loader, device, num_epochs=EPOCHS, lr=LR, save_path=MODEL)
    plot_training(history)

    # ── Cargar mejor modelo ───────────────────────────────────────────────────
    model.load_state_dict(torch.load(MODEL, map_location=device, weights_only=False))
    print(f"\n[*] Mejor modelo cargado: {MODEL}")

    # ── Recolectar scores en los conjuntos mixtos ─────────────────────────────
    print("\n[*] Recolectando scores en conjunto de grid-search...")
    C_gs, T_gs, R_gs, L_gs, M_gs, _ = collect_raw(model, gs_loader, device)

    print("[*] Recolectando scores en test...")
    C_te, T_te, R_te, L_te, M_te, _ = collect_raw(model, te_loader, device)
    
    qcd_gs  = L_gs == 0
    qcd_te  = L_te == 0

    # ==========================================================================
    # EL NUEVO CORAZÓN DEL MÉTODO: DDT INDEPENDIENTE (DUAL DDT)
    # ==========================================================================
    print("\n" + "="*50)
    print("  FASE DE PURIFICACIÓN (DUAL DDT)")
    print("="*50)

    # 1. Purificar el Autoencoder (Chamfer)
    ddt_C = DDTransform(name="Chamfer", poly_deg=3, quantile=DDT_QUANT)
    rc_C, qc_C, _ = ddt_C.fit(C_gs[qcd_gs], R_gs[qcd_gs], n_bins=20)
    C_gs_ddt = ddt_C.transform(C_gs, R_gs)
    ddt_C.save("ddt_chamfer_v11.json")

    # 2. Purificar la física experta (Tau21)
    ddt_T = DDTransform(name="Tau21", poly_deg=3, quantile=DDT_QUANT)
    rc_T, qc_T, _ = ddt_T.fit(T_gs[qcd_gs], R_gs[qcd_gs], n_bins=20)
    T_gs_ddt = ddt_T.transform(T_gs, R_gs)
    ddt_T.save("ddt_tau21_v11.json")

    # Diagnósticos visuales separados
    s_C_ddt_gs_qcd = ddt_C.transform(C_gs[qcd_gs], R_gs[qcd_gs])
    s_T_ddt_gs_qcd = ddt_T.transform(T_gs[qcd_gs], R_gs[qcd_gs])
    plot_ddt_fit(rc_C, qc_C, ddt_C, R_gs[qcd_gs], C_gs[qcd_gs], s_C_ddt_gs_qcd, "v11_ddt_fit_chamfer.png")
    plot_ddt_fit(rc_T, qc_T, ddt_T, R_gs[qcd_gs], T_gs[qcd_gs], s_T_ddt_gs_qcd, "v11_ddt_fit_tau21.png")

    # ==========================================================================
    # GRID SEARCH SOBRE VARIABLES LIMPIAS
    # ==========================================================================
    print("\n[*] Grid search sobre variables purificadas (Súper veloz)...")
    best_params, auc_val = grid_search_hybrid(C_gs_ddt, T_gs_ddt, L_gs)
    
    alpha = best_params['alpha']
    gamma = best_params['gamma']

    # ==========================================================================
    # EVALUACIÓN FINAL EN TEST
    # ==========================================================================
    # Aplicar DDT al test
    C_te_ddt = ddt_C.transform(C_te, R_te)
    T_te_ddt = ddt_T.transform(T_te, R_te)

    # Score final combinado
    s_te_raw = alpha * C_te     - gamma * T_te
    s_te_ddt = alpha * C_te_ddt - gamma * T_te_ddt

    s_dir, auc_test = fix_dir(s_te_ddt, L_te)
    print(f"\n[+] AUC final en TEST (Dual DDT) = {auc_test:.4f}")

    # Chequeo final de correlación global
    rho_after, _ = pearsonr(s_te_ddt[qcd_te], R_te[qcd_te])
    print(f"[*] Correlación global final en TEST: ρ(s_DDT, ρ) = {rho_after:+.4f}")

    # ── Panel completo de gráficas ────────────────────────────────────────────
    rej50, sig_max, jsd_vals = plot_all(C_te, T_te, s_te_raw, s_te_ddt, L_te, M_te, R_te, auc_test, save_prefix='v11')

    # ── Guardar resultados ────────────────────────────────────────────────────
    result = dict(
        auc_ddt          = float(auc_test),
        rej_at_50tpr     = float(rej50),
        max_significance = float(sig_max),
        jsd_1pct         = float(jsd_vals.get('1% QCD', 999)),
        alpha            = alpha,
        gamma            = gamma,
        rho_after_ddt    = float(rho_after),
    )
    with open("results_v11.json", "w") as f:
        json.dump(result, f, indent=2)
    print("\n[*] Resultados guardados en 'results_v11.json'")

if __name__ == "__main__":
    main()

[*] Device: cuda
[*] Fit preprocesador sobre 3 archivo(s)...
    feat_mean=[2.7726523e-03 5.9879196e-05 2.4905994e+00]

[*] Cargando 3 archivo(s) para Entrenamiento AE...
    AE Train=16261  AE Val=1807

[*] Cargando datos mixtos (Señal + Fondo) desde: mixed_signal_qcd.csv
    Grid Search=94609  Test=94610


Epoch   1/60: 100%|████████████████████████████████| 64/64 [00:03<00:00, 17.62it/s, Chamfer=0.50576]


  → Train=0.64093  Val=0.48613
  ✓ Checkpoint (val=0.48613)


Epoch   2/60: 100%|████████████████████████████████| 64/64 [00:02<00:00, 22.06it/s, Chamfer=0.44780]


  → Train=0.44799  Val=0.44841
  ✓ Checkpoint (val=0.44841)


...

  → Train=0.18200  Val=0.20716
  ✓ Checkpoint (val=0.20716)


Epoch  59/60: 100%|████████████████████████████████| 64/64 [00:02<00:00, 22.61it/s, Chamfer=0.18595]


  → Train=0.18096  Val=0.20700
  ✓ Checkpoint (val=0.20700)


Epoch  60/60: 100%|████████████████████████████████| 64/64 [00:02<00:00, 22.12it/s, Chamfer=0.21234]


  → Train=0.18261  Val=0.20717

[*] Mejor modelo cargado: best_ae_v11.pt

[*] Recolectando scores en conjunto de grid-search...
[*] Recolectando scores en test...

  FASE DE PURIFICACIÓN (DUAL DDT)
[*] Ajustando DDT para 'Chamfer' (deg=3, q=50)...
    -> Residual RMS: 0.02421
[*] Ajustando DDT para 'Tau21' (deg=3, q=50)...
    -> Residual RMS: 0.01850
[*] DDT fit guardado en 'v11_ddt_fit_chamfer.png'
[*] DDT fit guardado en 'v11_ddt_fit_tau21.png'

[*] Grid search sobre variables purificadas (Súper veloz)...

[+] Grid search (val) → AUC=0.8213  α=4.25  γ=3.25

[+] AUC final en TEST (Dual DDT) = 0.8201
[*] Correlación global final en TEST: ρ(s_DDT, ρ) = +0.0066

     FPR  |   JSD raw  |   JSD DDT  |  N events
  ---------------------------------------------
   50.0%  |    0.0012  |    0.0020  |     24007
   10.0%  |    0.0044  |    0.0019  |      4802
    5.0%  |    0.0065  |    0.0043  |      2401
    1.0%  |    0.0265  |    0.0238  |       481
    0.5%  |    0.0494  |    0.0463  |     